# Phase 4 — Model Optimisation, Ensemble & Analysis

**Learning Goals**
- Create a soft-voting ensemble of trained models
- Generate and interpret confusion matrix, ROC curve, classification report
- Visualise model decisions using Grad-CAM
- Build a multi-model comparison dashboard

**Tasks to complete:**
1. `TODO 8`  in `src/models.py`   — `build_ensemble()`
2. `TODO 12` in `src/evaluate.py` — `evaluate_model()`
3. `TODO 13` in `src/evaluate.py` — `plot_confusion_matrix()`
4. `TODO 14` in `src/evaluate.py` — `plot_roc_curve()`
5. `TODO 15` in `src/evaluate.py` — `grad_cam()`

Run `pytest tests/test_phase4.py -v` to check your work.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.data_loader import get_data_generators, compute_class_weights
from src.train import get_callbacks, train_model
from src.evaluate import evaluate_model, plot_confusion_matrix, plot_roc_curve, compare_models, grad_cam
from src.models import build_advanced_nn, build_transfer_model, build_ensemble

In [ ]:
train_gen, val_gen, test_gen = get_data_generators(
    data_dir='../data/chest_xray',
    img_size=(224, 224),
    batch_size=32,
)
class_weights = compute_class_weights(train_gen)

## 1. Load or Re-train Best Models

Load saved models from Phase 3 (or re-train here).

In [ ]:
import os

def load_or_train(model_path, build_fn, **build_kwargs):
    if os.path.exists(model_path):
        print(f'Loading {model_path}')
        return tf.keras.models.load_model(model_path)
    print(f'Training {model_path}')
    m = build_fn(**build_kwargs)
    cb = get_callbacks(model_name=os.path.basename(model_path), patience=5)
    train_model(m, train_gen, val_gen, epochs=15, class_weights=class_weights, callbacks=cb)
    m.save(model_path)
    return m

model_adv = load_or_train(
    '../models/advanced_nn_best.keras',
    build_advanced_nn,
    input_shape=(224, 224, 3),
)

model_tl = load_or_train(
    '../models/mobilenetv2_best.keras',
    build_transfer_model,
    base_model_name='MobileNetV2',
    input_shape=(224, 224, 3),
    freeze_base=False,
)

## 2. Ensemble

In [ ]:
# TODO: implement build_ensemble() in src/models.py
ensemble = build_ensemble([model_adv, model_tl], input_shape=(224, 224, 3))
print('Ensemble output shape:', ensemble.output_shape)

## 3. Full Evaluation

In [ ]:
# TODO: implement evaluate_model() in src/evaluate.py
all_results = {}
for name, m in [('Advanced NN', model_adv), ('MobileNetV2', model_tl), ('Ensemble', ensemble)]:
    print(f'Evaluating {name}...')
    all_results[name] = evaluate_model(m, test_gen)
    print(f"  Accuracy: {all_results[name]['accuracy']:.4f}  AUC: {all_results[name]['auc']:.4f}")

In [ ]:
# Confusion matrix for ensemble
# TODO: implement plot_confusion_matrix() in src/evaluate.py
plot_confusion_matrix(all_results['Ensemble']['confusion_matrix'])

In [ ]:
# ROC curves
# TODO: implement plot_roc_curve() in src/evaluate.py
import numpy as np

# Collect true labels from test generator
y_true = np.concatenate([y for _, y in test_gen])

for name, m in [('Advanced NN', model_adv), ('MobileNetV2', model_tl), ('Ensemble', ensemble)]:
    y_probs = m.predict(test_gen, verbose=0).squeeze()
    plot_roc_curve(y_true, y_probs, model_name=name)

In [ ]:
# Multi-model comparison bar chart
compare_models(all_results)

## 4. Grad-CAM Visualisation

Implement `grad_cam()` in `src/evaluate.py`, then visualise which regions activate for each prediction.

In [ ]:
# TODO: implement grad_cam() in src/evaluate.py

import cv2

# Pick a sample test image
test_imgs, test_labels = next(test_gen)
sample_img = test_imgs[0:1]   # shape (1, 224, 224, 3)

# Find the last conv layer name in MobileNetV2 base
base = next(l for l in model_tl.layers if hasattr(l, 'layers') and len(l.layers) > 5)
last_conv_name = [l.name for l in base.layers if 'conv' in l.name.lower()][-1]
print('Last conv layer:', last_conv_name)

# Compute and overlay heatmap
# NOTE: for sub-model conv layers, you may need to pass the base sub-model instead
heatmap = grad_cam(model_tl, sample_img, last_conv_layer_name=last_conv_name)

heatmap_resized = cv2.resize(heatmap, (224, 224))
heatmap_color   = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
heatmap_color   = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

superimposed = (heatmap_color * 0.4 + sample_img[0] * 255 * 0.6).clip(0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(sample_img[0]); axes[0].set_title('Original')
axes[1].imshow(heatmap_resized, cmap='jet'); axes[1].set_title('Grad-CAM Heatmap')
axes[2].imshow(superimposed); axes[2].set_title('Superimposed')
for ax in axes: ax.axis('off')
plt.suptitle(f'True: {"PNEUMONIA" if test_labels[0] == 1 else "NORMAL"}')
plt.tight_layout()
plt.show()

## 5. Reflection Questions

1. Does the ensemble always outperform individual models? Explain why or why not.
2. What does the confusion matrix reveal about the model's clinical reliability? Which error type (false positive or false negative) is more costly in pneumonia diagnosis?
3. What do the Grad-CAM regions tell you about what the model has learned?
4. If this model were deployed in a hospital, what safeguards would you recommend?

**Your answers here:**

1. 
2. 
3. 
4. 